# BÀI 1. THỰC HÀNH XÂY DỰNG MÔ HÌNH CLASSIFICATION VỚI DỮ LIỆU BẢNG


# PHẦN 1. CHUẨN BỊ MÔI TRƯỜNG VÀ DỮ LIỆU GỐC


1. Import thư viện: Nạp numpy, pandas, matplotlib, các công cụ tiền xử lý, chia tập, huấn luyện và đánh giá từ scikit-learn cùng các kỹ thuật lấy mẫu "RandomOverSampler, RandomUnderSampler, SMOTE" từ imbalanced-learn.
2. Chuẩn bị dữ liệu: Nạp bộ dữ liệu Breast Cancer Wisconsin, quy ước nhãn 1 = malignant "ác tính" và 0 = benign "lành tính", sau đó tạo tập dữ liệu   thực hành mất cân bằng mặc định gồm toàn bộ mẫu benign "357 mẫu" và 60 mẫu malignant.
3. Chia tập Train/Test: Tách dữ liệu theo tỷ lệ 80% Train – 20% Test "random_state=42, stratify=y" để giữ nguyên tỷ lệ phân bố lớp.
4. Hàm và Pipeline cơ sở: Định nghĩa bộ chỉ số đánh giá scoring "Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC", hàm tổng hợp cv_summary, chiến lược chia StratifiedKFold "n_splits=5" và mô hình cơ sở LogisticRegression kết hợp StandardScaler.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    cross_validate
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    make_scorer
)

from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42

cancer = load_breast_cancer(as_frame=True)
X_full = cancer.data.copy()
y_full = (cancer.target == 0).astype(int)

df_full = X_full.copy()
df_full["target"] = y_full

benign_df = df_full[df_full["target"] == 0]
malignant_df = df_full[df_full["target"] == 1].sample(
    n=60,
    random_state=RANDOM_STATE
)

df = pd.concat([benign_df, malignant_df], axis=0)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

def cv_summary(model, X_data, y_data, cv):
    scores = cross_validate(
        model,
        X_data,
        y_data,
        cv=cv,
        scoring=scoring
    )

    row = {}
    for metric in scoring.keys():
        values = scores[f"test_{metric}"]
        row[f"{metric}_mean"] = values.mean()
        row[f"{metric}_std"] = values.std()

    return row

stratified_kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

baseline_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=5000,
        random_state=RANDOM_STATE
    ))
])


# PHẦN 2. BÀI TẬP THỰC HÀNH

## Bài 1 — Mức độ imbalance
So sánh `Accuracy`, `Recall`, `F1` và `PR-AUC` khi thay số mẫu `malignant` thành `40`, `120` và toàn bộ dữ liệu gốc.


### Trường hợp 1.1: Thay `n=60` thành `n=40`


In [2]:
benign_df = df_full[df_full["target"] == 0]
malignant_df = df_full[df_full["target"] == 1].sample(
    n=40,
    random_state=RANDOM_STATE
)

df = pd.concat([benign_df, malignant_df], axis=0)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

X_b1 = df.drop(columns="target")
y_b1 = df["target"]
display(pd.Series(cv_summary(baseline_pipeline, X_b1, y_b1, stratified_kfold), name="n=40")[["accuracy_mean", "recall_mean", "f1_mean", "pr_auc_mean"]].round(4))


accuracy_mean    0.9949
recall_mean      0.9500
f1_mean          0.9714
pr_auc_mean      0.9917
Name: n=40, dtype: float64

### Trường hợp 1.2: Thay `n=60` thành `n=120`


In [3]:
benign_df = df_full[df_full["target"] == 0]
malignant_df = df_full[df_full["target"] == 1].sample(
    n=120,
    random_state=RANDOM_STATE
)

df = pd.concat([benign_df, malignant_df], axis=0)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

X_b1 = df.drop(columns="target")
y_b1 = df["target"]
display(pd.Series(cv_summary(baseline_pipeline, X_b1, y_b1, stratified_kfold), name="n=120")[["accuracy_mean", "recall_mean", "f1_mean", "pr_auc_mean"]].round(4))


accuracy_mean    0.9769
recall_mean      0.9250
f1_mean          0.9523
pr_auc_mean      0.9858
Name: n=120, dtype: float64

### Trường hợp 1.3: Dùng toàn bộ dữ liệu gốc (`n=212`)


In [4]:
benign_df = df_full[df_full["target"] == 0]
malignant_df = df_full[df_full["target"] == 1]

df = pd.concat([benign_df, malignant_df], axis=0)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

X_b1 = df.drop(columns="target")
y_b1 = df["target"]
display(pd.Series(cv_summary(baseline_pipeline, X_b1, y_b1, stratified_kfold), name="Toàn bộ dữ liệu")[["accuracy_mean", "recall_mean", "f1_mean", "pr_auc_mean"]].round(4))


accuracy_mean    0.9807
recall_mean      0.9623
f1_mean          0.9738
pr_auc_mean      0.9950
Name: Toàn bộ dữ liệu, dtype: float64

**Nhận xét Bài 1:**
* Khi thay `n=40`, dữ liệu bị mất cân bằng nghiêm trọng (40/397 mẫu), mô hình dễ bỏ sót mẫu `malignant` (`Recall` và `PR-AUC` giảm trong khi `Accuracy` vẫn cao giả tạo).
* Khi tăng lên `n=120` và toàn bộ dữ liệu gốc (`n=212`), dữ liệu cân bằng hơn giúp `Recall`, `F1` và `PR-AUC` tăng cao và ổn định hơn.


## Bài 2 — KFold và StratifiedKFold
So sánh `mean score` và `standard deviation` giữa các fold.


### Trường hợp 2.1: `KFold`


In [5]:
kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

display(pd.Series(cv_summary(baseline_pipeline, X_train, y_train, kfold), name="KFold").round(4))


accuracy_mean     0.9880
accuracy_std      0.0112
precision_mean    1.0000
precision_std     0.0000
recall_mean       0.9012
recall_std        0.1052
f1_mean           0.9446
f1_std            0.0613
roc_auc_mean      0.9975
roc_auc_std       0.0042
pr_auc_mean       0.9891
pr_auc_std        0.0188
Name: KFold, dtype: float64

### Trường hợp 2.2: `StratifiedKFold`


In [6]:
stratified_kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

display(pd.Series(cv_summary(baseline_pipeline, X_train, y_train, stratified_kfold), name="StratifiedKFold").round(4))


accuracy_mean     0.9850
accuracy_std      0.0164
precision_mean    0.9618
precision_std     0.0469
recall_mean       0.9400
recall_std        0.1200
f1_mean           0.9447
f1_std            0.0646
roc_auc_mean      0.9951
roc_auc_std       0.0098
pr_auc_mean       0.9855
pr_auc_std        0.0291
Name: StratifiedKFold, dtype: float64

**Nhận xét Bài 2:**
* `StratifiedKFold` có độ lệch chuẩn (`std`) thấp hơn so với `KFold`.
* Do dữ liệu bị mất cân bằng, `KFold` chia ngẫu nhiên làm tỷ lệ lớp dương giữa các fold không đều, còn `StratifiedKFold` giữ tỷ lệ lớp đồng nhất ở mọi fold nên kết quả ổn định hơn.


## Bài 3 — Thay đổi số fold
Quan sát sự thay đổi của `mean` và `standard deviation` với `K = 3`, `5`, `10`.


### Trường hợp 3.1: `K = 3`


In [7]:
stratified_kfold = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

display(pd.Series(cv_summary(baseline_pipeline, X_train, y_train, stratified_kfold), name="K = 3").round(4))


accuracy_mean     0.9880
accuracy_std      0.0042
precision_mean    1.0000
precision_std     0.0000
recall_mean       0.9167
recall_std        0.0295
f1_mean           0.9563
f1_std            0.0162
roc_auc_mean      0.9971
roc_auc_std       0.0040
pr_auc_mean       0.9907
pr_auc_std        0.0132
Name: K = 3, dtype: float64

### Trường hợp 3.2: `K = 5`


In [8]:
stratified_kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

display(pd.Series(cv_summary(baseline_pipeline, X_train, y_train, stratified_kfold), name="K = 5").round(4))


accuracy_mean     0.9850
accuracy_std      0.0164
precision_mean    0.9618
precision_std     0.0469
recall_mean       0.9400
recall_std        0.1200
f1_mean           0.9447
f1_std            0.0646
roc_auc_mean      0.9951
roc_auc_std       0.0098
pr_auc_mean       0.9855
pr_auc_std        0.0291
Name: K = 5, dtype: float64

### Trường hợp 3.3: `K = 10`


In [9]:
stratified_kfold = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=RANDOM_STATE
)

display(pd.Series(cv_summary(baseline_pipeline, X_train, y_train, stratified_kfold), name="K = 10").round(4))


accuracy_mean     0.9880
accuracy_std      0.0147
precision_mean    0.9667
precision_std     0.0667
recall_mean       0.9550
recall_std        0.0907
f1_mean           0.9564
f1_std            0.0550
roc_auc_mean      0.9957
roc_auc_std       0.0129
pr_auc_mean       0.9861
pr_auc_std        0.0417
Name: K = 10, dtype: float64

**Nhận xét Bài 3:**
* Khi tăng `K` từ `3` lên `5` và `10`, tập train mỗi fold lớn hơn nên điểm trung bình (`mean`) tăng nhẹ.
* Tuy nhiên ở `K = 10`, mỗi fold validation chỉ còn vài mẫu `malignant`, nên chỉ cần sai lệch 1 mẫu cũng làm `Recall` biến động lớn, dẫn đến `recall_std` tăng.


## Bài 4 — Thay đổi mức Over-Sampling
Giải thích trade-off giữa `Precision` và `Recall` với các mức `sampling_strategy`: `0.5`, `0.75`, `1.0`.


### Trường hợp 4.1: `sampling_strategy=0.5`


In [10]:
stratified_kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

model_ros_05 = ImbPipeline([
    ("scaler", StandardScaler()),
    ("sampler", RandomOverSampler(
        sampling_strategy=0.5,
        random_state=RANDOM_STATE
    )),
    ("model", LogisticRegression(
        max_iter=5000,
        random_state=RANDOM_STATE
    ))
])

display(pd.Series(cv_summary(model_ros_05, X_train, y_train, stratified_kfold), name="ROS 0.5")[["accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "pr_auc_mean"]].round(4))


accuracy_mean     0.9880
precision_mean    0.9618
recall_mean       0.9600
f1_mean           0.9577
pr_auc_mean       0.9836
Name: ROS 0.5, dtype: float64

### Trường hợp 4.2: `sampling_strategy=0.75`


In [11]:
model_ros_075 = ImbPipeline([
    ("scaler", StandardScaler()),
    ("sampler", RandomOverSampler(
        sampling_strategy=0.75,
        random_state=RANDOM_STATE
    )),
    ("model", LogisticRegression(
        max_iter=5000,
        random_state=RANDOM_STATE
    ))
])

display(pd.Series(cv_summary(model_ros_075, X_train, y_train, stratified_kfold), name="ROS 0.75")[["accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "pr_auc_mean"]].round(4))


accuracy_mean     0.9880
precision_mean    0.9618
recall_mean       0.9600
f1_mean           0.9577
pr_auc_mean       0.9855
Name: ROS 0.75, dtype: float64

### Trường hợp 4.3: `sampling_strategy=1.0`


In [12]:
model_ros_10 = ImbPipeline([
    ("scaler", StandardScaler()),
    ("sampler", RandomOverSampler(
        sampling_strategy=1.0,
        random_state=RANDOM_STATE
    )),
    ("model", LogisticRegression(
        max_iter=5000,
        random_state=RANDOM_STATE
    ))
])

display(pd.Series(cv_summary(model_ros_10, X_train, y_train, stratified_kfold), name="ROS 1.0")[["accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "pr_auc_mean"]].round(4))


accuracy_mean     0.9850
precision_mean    0.9418
recall_mean       0.9600
f1_mean           0.9472
pr_auc_mean       0.9855
Name: ROS 1.0, dtype: float64

**Nhận xét Bài 4:**
* Khi tăng `sampling_strategy` từ `0.5` lên `1.0`, số mẫu lớp thiểu số được nhân bản nhiều hơn giúp `Recall` tăng lên.
* **Trade-off:** Ngược lại, việc ưu tiên lớp dương làm tăng báo động nhầm (False Positive) nên `Precision` giảm nhẹ.


## Bài 5 — So sánh ba kỹ thuật sampling
So sánh `Random Over-Sampling`, `Random Under-Sampling` và `SMOTE` qua `Precision`, `Recall`, `F1` và `PR-AUC`.


### Trường hợp 5.1: `Random OverSampling`


In [13]:
model_ros = ImbPipeline([
    ("scaler", StandardScaler()),
    ("sampler", RandomOverSampler(
        random_state=RANDOM_STATE
    )),
    ("model", LogisticRegression(
        max_iter=5000,
        random_state=RANDOM_STATE
    ))
])

display(pd.Series(cv_summary(model_ros, X_train, y_train, stratified_kfold), name="Random OverSampling")[["accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "pr_auc_mean"]].round(4))


accuracy_mean     0.9850
precision_mean    0.9418
recall_mean       0.9600
f1_mean           0.9472
pr_auc_mean       0.9855
Name: Random OverSampling, dtype: float64

### Trường hợp 5.2: `Random UnderSampling`


In [14]:
model_rus = ImbPipeline([
    ("scaler", StandardScaler()),
    ("sampler", RandomUnderSampler(
        random_state=RANDOM_STATE
    )),
    ("model", LogisticRegression(
        max_iter=5000,
        random_state=RANDOM_STATE
    ))
])

display(pd.Series(cv_summary(model_rus, X_train, y_train, stratified_kfold), name="Random UnderSampling")[["accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "pr_auc_mean"]].round(4))


accuracy_mean     0.9849
precision_mean    0.9255
recall_mean       0.9800
f1_mean           0.9494
pr_auc_mean       0.9895
Name: Random UnderSampling, dtype: float64

### Trường hợp 5.3: `SMOTE`


In [15]:
model_smote = ImbPipeline([
    ("scaler", StandardScaler()),
    ("sampler", SMOTE(
        random_state=RANDOM_STATE
    )),
    ("model", LogisticRegression(
        max_iter=5000,
        random_state=RANDOM_STATE
    ))
])

display(pd.Series(cv_summary(model_smote, X_train, y_train, stratified_kfold), name="SMOTE")[["accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "pr_auc_mean"]].round(4))


accuracy_mean     0.9880
precision_mean    0.9618
recall_mean       0.9600
f1_mean           0.9577
pr_auc_mean       0.9836
Name: SMOTE, dtype: float64

**Nhận xét Bài 5:**
* Cả 3 kỹ thuật có `Accuracy` gần bằng nhau (~98.5% – 98.8%).
* `Random UnderSampling` cho `Recall` (0.9800) và `PR-AUC` (0.9895) cao nhất nhưng mất dữ liệu lớp đa số nên `Precision` thấp nhất (0.9255).
* `Random OverSampling` tăng `Recall` lên 0.9600 nhưng sao chép mẫu cũ khiến `Precision` giảm còn 0.9418.
* `SMOTE` tạo mẫu tổng hợp mới nên giữ `Precision` cao (0.9618) và `Recall` 0.9600, đạt `F1` cao nhất (0.9577).


## Bài 6 — Thay classifier
Thay `Logistic Regression` bằng `KNN`, `SVM`, `Decision Tree` và phân tích độ nhạy với `StandardScaler`.


### Trường hợp 6.1: `KNN`


In [16]:
knn_no_scale = ImbPipeline([
    ("sampler", SMOTE(random_state=RANDOM_STATE)),
    ("model", KNeighborsClassifier())
])

knn_scaled = ImbPipeline([
    ("scaler", StandardScaler()),
    ("sampler", SMOTE(random_state=RANDOM_STATE)),
    ("model", KNeighborsClassifier())
])

scale_compare = pd.DataFrame({
    "KNN - Raw": cv_summary(knn_no_scale, X_train, y_train, stratified_kfold),
    "KNN - StandardScaler": cv_summary(knn_scaled, X_train, y_train, stratified_kfold)
}).T

display(scale_compare[["accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "pr_auc_mean"]].round(4))


,accuracy_mean,precision_mean,recall_mean,f1_mean,pr_auc_mean
KNN - Raw,0.9429,0.7607,0.8978,0.8191,0.8588
KNN - StandardScaler,0.9730,0.8833,0.9600,0.9128,0.9442


### Trường hợp 6.2: `SVM`


In [17]:
svm_no_scale = ImbPipeline([
    ("sampler", SMOTE(random_state=RANDOM_STATE)),
    ("model", SVC(probability=True, random_state=RANDOM_STATE))
])

svm_scaled = ImbPipeline([
    ("scaler", StandardScaler()),
    ("sampler", SMOTE(random_state=RANDOM_STATE)),
    ("model", SVC(probability=True, random_state=RANDOM_STATE))
])

scale_compare = pd.DataFrame({
    "SVM - Raw": cv_summary(svm_no_scale, X_train, y_train, stratified_kfold),
    "SVM - StandardScaler": cv_summary(svm_scaled, X_train, y_train, stratified_kfold)
}).T

display(scale_compare[["accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "pr_auc_mean"]].round(4))


,accuracy_mean,precision_mean,recall_mean,f1_mean,pr_auc_mean
SVM - Raw,0.9609,0.9124,0.8111,0.8544,0.9159
SVM - StandardScaler,0.9880,0.9618,0.9600,0.9577,0.9807


### Trường hợp 6.3: `Decision Tree`


In [18]:
dt_no_scale = ImbPipeline([
    ("sampler", SMOTE(random_state=RANDOM_STATE)),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
])

dt_scaled = ImbPipeline([
    ("scaler", StandardScaler()),
    ("sampler", SMOTE(random_state=RANDOM_STATE)),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
])

scale_compare = pd.DataFrame({
    "Decision Tree - Raw": cv_summary(dt_no_scale, X_train, y_train, stratified_kfold),
    "Decision Tree - StandardScaler": cv_summary(dt_scaled, X_train, y_train, stratified_kfold)
}).T

display(scale_compare[["accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "pr_auc_mean"]].round(4))


,accuracy_mean,precision_mean,recall_mean,f1_mean,pr_auc_mean
Decision Tree - Raw,0.946,0.8034,0.8556,0.8165,0.7184
Decision Tree - StandardScaler,0.973,0.8975,0.9400,0.9117,0.8484


**Nhận xét Bài 6:**
* **KNN và SVM rất nhạy với `StandardScaler`:** Do tính khoảng cách hình học, khi chưa chuẩn hóa các cột có giá trị lớn áp đảo hoàn toàn các cột nhỏ làm `F1` rất thấp; sau khi dùng `StandardScaler`, hiệu năng tăng vọt.
* **Decision Tree không nhạy với `StandardScaler`:** Cây quyết định chia nhánh theo ngưỡng giá trị từng cột độc lập nên thang đo không ảnh hưởng đến kết quả.


## Bài 7 — Câu hỏi suy luận

1. **Tại sao không nên sampling tập test?**
   * Tập test đại diện cho dữ liệu thực tế chưa từng thấy, cần giữ nguyên phân phối tự nhiên để phản ánh đúng hiệu năng thật khi triển khai.

2. **Tại sao không nên fit StandardScaler trên toàn bộ dữ liệu?**
   * Fit scaler trên toàn bộ dữ liệu làm rò rỉ thông tin thống kê (mean, std) của tập validation/test vào quá trình huấn luyện (**data leakage**).

3. **Khi nào Recall quan trọng hơn Precision?**
   * Khi việc bỏ sót ca dương tính (False Negative) gây hậu quả nghiêm trọng, ví dụ: chẩn đoán bệnh ung thư, phát hiện gian lận tài chính.

4. **Khi nào Precision quan trọng hơn Recall?**
   * Khi việc dự đoán nhầm sang dương tính (False Positive) gây tốn kém hoặc phiền toái lớn, ví dụ: lọc thư rác (tránh nhầm thư quan trọng) hoặc khóa tài khoản nghi ngờ vi phạm.

5. **Vì sao PR-AUC thường hữu ích với dữ liệu imbalance?**
   * PR-AUC chỉ tập trung đánh giá `Precision` và `Recall` trên lớp thiểu số (Positive), không bị làm đẹp giả tạo bởi số lượng lớn mẫu âm tính (True Negative) như ROC-AUC.

6. **Vì sao `StratifiedKFold` phù hợp hơn `KFold` trong bài này?**
   * Vì dữ liệu lệch lớp (lớp dương chỉ chiếm ~14.4%); `StratifiedKFold` đảm bảo tỷ lệ hai lớp đồng đều ở tất cả các fold giúp kết quả đánh giá ổn định.

7. **Random Under-Sampling có thể làm mất thông tin như thế nào?**
   * Vì nó xóa ngẫu nhiên phần lớn mẫu thuộc lớp đa số (`Benign`), có thể làm mất các điểm dữ liệu quan trọng nằm gần ranh giới phân loại.

8. **SMOTE khác Random Over-Sampling ở điểm nào?**
   * `Random Over-Sampling` sao chép y hệt các mẫu thiểu số có sẵn (dễ gây overfitting), còn `SMOTE` nội suy từ các láng giềng gần nhất để **sinh ra các mẫu tổng hợp mới**.
